# NB6 — パラメータ依存性（第13章; 要求仕様の定量分析）

**目標**: $\tau$（再電離）と $z_*$（再結合シフト）の $\mathcal D_\ell$ への影響を 「どの方向に・何%・なぜ」で示す。$\sum m_\nu$ は本版では CLASS 委譲（04 §7）。所要時間: <1分（キャッシュ利用）。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from cmbcore.plotstyle import use_style; use_style()
np.random.seed(5220)
FIG = Path('..') / 'figures'
import json
d = np.load(FIG/'param_study.npz'); ells = d['ells']
S = json.loads((FIG/'param_study_summary.json').read_text())
print(json.dumps(S, indent=2))

## F6.4 光学的厚み $\tau$（再電離 $z_{\rm re}=8$）: $\ell\gtrsim30$ で $e^{-2\tau}$ 抑制

In [ ]:
plt.plot(ells, d['fid'], label='no reion')
plt.plot(ells, d['reio_z8'], label='reion z_re=8')
plt.xlabel('l'); plt.ylabel('D_l [uK^2]'); plt.legend()
plt.title('tau~%.3f: suppression ~%.1f%% at l=500' % (S['tau_reio_z8'], S['reio_suppression_l500_pct'])); plt.show()
print('predicted e^{-2tau}-1 = %.1f%%' % S['reio_predicted_exp_-2tau_pct'])

## F6.6 再結合シフト ($T_b\to1.05\,T_b$): $z_*$ と $\ell_1$ の応答

In [ ]:
plt.plot(ells, d['fid'], label='fiducial (z_*=%.0f)'%S['z_star_fid'])
plt.plot(ells, d['recomb_p5'], label='shift +5%% (z_*=%.0f)'%S['z_star_shift_p5'])
plt.xlabel('l'); plt.ylabel('D_l [uK^2]'); plt.xlim(2,800); plt.legend(); plt.show()
print('l1: %d -> %d' % (S['l1_fid'], S['l1_recomb_p5']))

## F6.7 ニュートリノ質量 $\sum m_\nu$（背景を厳密実装; $h$ 固定）
`Params(Sigma_mnu=...)` で massive ν 背景（運動量積分）が有効になり、$\Omega_\Lambda$ 減・$\theta_*$ 変化として TT に効く。

In [ ]:
mf = FIG/'massive_nu.npz'
if mf.exists():
    m = np.load(mf); el = m['ells']
    plt.plot(el, m['core_0.12']/m['core_0.0'], label='cmbcore ratio')
    if 'class_0.12' in m: plt.plot(el, m['class_0.12']/m['class_0.0'],'--',label='CLASS ratio')
    plt.axhline(1,c='k',lw=.6); plt.xlabel('l')
    plt.ylabel('C_l(0.12)/C_l(0)'); plt.legend(); plt.title('Sm_nu=0.12 eV TT response'); plt.show()
else:
    print('run scripts/make_massive_nu.py for massive_nu.npz')

## 観測への含意
- **$\tau$**: 再電離で散乱が増え、$\ell\gtrsim30$ の全ピークが一様に $e^{-2\tau}$ で抑制される（低 $\ell$ は再電離バンプで例外）。
- **$z_*$**: 再結合が早まると音地平線 $r_s$ が縮み、$\ell_1\propto1/\theta_*$ が高 $\ell$ 側へ動く。
- **$\sum m_\nu$**: 背景（$H(z),\theta_*$, 後期ISW）が支配的で TT への効果は小さい。本実装は背景を厳密に扱い CLASS と整合（摂動の自由流＝$P(k)$ 抑制は次段）。
## 【課題】 `Params(z_reio=...)` を変え $\tau$ と抑制率の関係を確認せよ。